# Configuracion Inicial

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Practica_DataFrames").getOrCreate()

### Ejercicio 1

In [5]:
datos = [
    ("Ana", 28, "Ventas", 35000),
    ("Juan", 35, "IT", 45000),
    ("María", 42, "IT", 55000),
    ("Pedro", 31, "Ventas", 38000),
    ("Laura", 29, "Marketing", 40000)
]
columnas = ["nombre", "edad", "departamento", "salario"]

df_emp = spark.createDataFrame(datos, columnas)

df_emp.printSchema()
df_emp.show(3)
print(f"Total de empleados: {df_emp.count()}")

root
 |-- nombre: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+
only showing top 3 rows
Total de empleados: 5


### Ejercicio 2

In [6]:
# IT con salario > 40000
df_emp.filter((F.col("departamento") == "IT") & (F.col("salario") > 40000)).show()

# Ventas O Marketing
df_emp.filter((F.col("departamento") == "Ventas") | (F.col("departamento") == "Marketing")).show()

# NO son de IT
df_emp.filter(~(F.col("departamento") == "IT")).show()

# Entre 30 y 40 años
df_emp.filter((F.col("edad") >= 30) & (F.col("edad") <= 40)).show()

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| Pedro|  31|      Ventas|  38000|
+------+----+------------+-------+



### Ejercicio 3

In [7]:
df_ej3 = (
    df_emp
    .withColumn("salario_en_5_años", F.col("salario") + (F.col("salario") * 0.05 * 5))
    .withColumnRenamed("nombre", "empleado")
)
df_ej3.show()

+--------+----+------------+-------+-----------------+
|empleado|edad|departamento|salario|salario_en_5_años|
+--------+----+------------+-------+-----------------+
|     Ana|  28|      Ventas|  35000|          43750.0|
|    Juan|  35|          IT|  45000|          56250.0|
|   María|  42|          IT|  55000|          68750.0|
|   Pedro|  31|      Ventas|  38000|          47500.0|
|   Laura|  29|   Marketing|  40000|          50000.0|
+--------+----+------------+-------+-----------------+



### Ejercicio 4

In [8]:
df_emp.groupBy("departamento").agg(
    F.avg("salario").alias("salario_promedio"),
    F.max("salario").alias("salario_maximo"),
    F.min("salario").alias("salario_minimo"),
    F.count("nombre").alias("total_empleados"),
    F.sum("salario").alias("gasto_total")
).orderBy(F.desc("gasto_total")).show()

+------------+----------------+--------------+--------------+---------------+-----------+
|departamento|salario_promedio|salario_maximo|salario_minimo|total_empleados|gasto_total|
+------------+----------------+--------------+--------------+---------------+-----------+
|          IT|         50000.0|         55000|         45000|              2|     100000|
|      Ventas|         36500.0|         38000|         35000|              2|      73000|
|   Marketing|         40000.0|         40000|         40000|              1|      40000|
+------------+----------------+--------------+--------------+---------------+-----------+



### Ejercicio 5

In [9]:
ventas_mensuales = [
    ("Enero", "ProdA", 100),
    ("Enero", "ProdB", 150),
    ("Febrero", "ProdA", 120)
]
df_ventas_mes = spark.createDataFrame(ventas_mensuales, ["mes", "producto", "ventas"])

ventana = Window.partitionBy("mes").orderBy(F.col("ventas").desc())

df_ventas_mes.withColumn("ranking", F.rank().over(ventana)).show()

+-------+--------+------+-------+
|    mes|producto|ventas|ranking|
+-------+--------+------+-------+
|  Enero|   ProdB|   150|      1|
|  Enero|   ProdA|   100|      2|
|Febrero|   ProdA|   120|      1|
+-------+--------+------+-------+



### Ejercicio 6

In [10]:
df_personas = spark.createDataFrame([("Ana", 1), ("Luis", 2), ("Marta", 3)], ["Nombre", "DepID"])
df_deptos = spark.createDataFrame([(1, "IT"), (2, "HR")], ["DepID", "NombreDep"])

df_personas.join(df_deptos, "DepID", "left").show()

+-----+------+---------+
|DepID|Nombre|NombreDep|
+-----+------+---------+
|    1|   Ana|       IT|
|    2|  Luis|       HR|
|    3| Marta|     NULL|
+-----+------+---------+



### Ejercicio 7

In [11]:
df_emp.createOrReplaceTempView("empleados")

spark.sql("""
    SELECT *
    FROM empleados
    WHERE departamento = 'IT' AND salario > 40000
""").show()

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+



### Ejercicio 8

In [12]:
df_nombres = spark.createDataFrame([("Juan", "García"), ("Ana", " Pérez ")], ["nombre", "apellido"])

df_limpio = (
    df_nombres
    .withColumn("nombre", F.trim(F.col("nombre")))
    .withColumn("apellido", F.trim(F.col("apellido")))
    .withColumn("Nombre_Completo", F.concat_ws(" ", F.col("nombre"), F.col("apellido")))
)
df_limpio.show(truncate=False)

+------+--------+---------------+
|nombre|apellido|Nombre_Completo|
+------+--------+---------------+
|Juan  |García  |Juan García    |
|Ana   |Pérez   |Ana Pérez      |
+------+--------+---------------+

